# 09 · Dates and time series

**Session 8 — Space and Time** · about 35 minutes

**You will learn to**
- parse dates explicitly and build datetimes from separate columns
- resample and smooth time series to the scale of your question
- find gaps and weekly patterns instead of hiding them
- handle time zones with `zoneinfo`

**Data:** COVID-19 doses per day (Transparència Catalunya), traffic accidents 2024 (Open Data BCN).

**Outline**
0. Before you start (no AI)
1. Parsing dates
2. Resampling and smoothing
3. Gaps and patterns
4. Datetimes from parts
5. Time zones
6. Space and time
7. Fitting (optional)
8. Your data

> **AI guardrails** — *Think first · Own it · Verify · Disclose · Protect.* AI tools are allowed in work time, but answer the **No-AI** questions on paper first, and make every ✅ self-check pass with code you can explain line by line.

## 0. Before you start (no AI)

1. A daily series covers 2021-01-01 to 2021-12-31. How many rows does `s.resample("W").sum()` have, roughly?
2. Vaccination centres close on Sundays. What will a daily plot look like? What would you plot instead?

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
from pathlib import Path

DATA = Path("../data")
OUTPUTS = Path("../outputs")
OUTPUTS.mkdir(exist_ok=True)
assert DATA.exists(), "Start Jupyter from the repository folder: notebooks/ must sit next to data/."

## 1. Parsing dates

The API returns dates as text like `2021-03-15T00:00:00.000`. Parse them explicitly:

In [ ]:
raw = pd.read_parquet(DATA / "examples/covid_vaccination_daily.parquet")
print(raw.dtypes)
raw["date"] = pd.to_datetime(raw["data"], format="ISO8601")
daily = raw.groupby("date")["dosis"].sum()
daily.head()

## 2. Resampling and smoothing

Daily counts are noisy (weekends, holidays). Aggregate to the scale of your question, or smooth.

In [ ]:
weekly = daily.resample("W").sum()
rolling = daily.rolling(7, center=True).mean()

fig, axs = plt.subplots(3, 1, figsize=(9, 7), sharex=True, layout="constrained")
axs[0].plot(daily.index, daily, lw=0.6)
axs[0].set_title("Daily: jagged, dominated by the weekly cycle")
axs[1].bar(weekly.index, weekly, width=6)
axs[1].set_title("Weekly totals")
axs[2].plot(rolling.index, rolling)
axs[2].set_title("7-day centred rolling mean")
for ax in axs:
    ax.set_ylabel("doses")
plt.show()

## 3. Gaps and patterns

`asfreq("D")` puts every calendar day in the index, so missing days become visible (NaN) instead of
silently disappearing from a line plot.

In [ ]:
full = daily.asfreq("D")
missing = full[full.isna()].index
print(f"{len(missing)} days without data between {full.index.min():%Y-%m-%d} and {full.index.max():%Y-%m-%d}")
missing[:10]

In [ ]:
in_2021 = daily["2021"]
weekday_mean = in_2021.groupby(in_2021.index.day_name()).mean().reindex(
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"])
ax = weekday_mean.plot.bar(figsize=(6, 3), color="tab:blue")
ax.set_ylabel("mean doses per day (2021)")
ax.set_xlabel(None)
plt.show()

## 4. Datetimes from separate columns

Barcelona accidents give year, month, day and hour in separate text columns. Build one datetime:

In [ ]:
acc = pd.read_parquet(DATA / "examples/bcn_accidents_2024.parquet").drop_duplicates("Numero_expedient")
parts = acc[["NK_ Any", "Mes_ any", "Dia_mes", "Hora_dia"]].apply(pd.to_numeric)
parts.columns = ["year", "month", "day", "hour"]
acc["when"] = pd.to_datetime(parts)
acc["weekend"] = acc["when"].dt.dayofweek >= 5

profile = acc.groupby(["weekend", acc["when"].dt.hour]).size().unstack(0)
profile = profile.div([acc["weekend"].eq(False).sum(), acc["weekend"].sum()], axis=1) * 100
ax = profile.plot(figsize=(7, 3.5), marker="o")
ax.set_xlabel("hour of the day")
ax.set_ylabel("% of accidents")
ax.legend(["weekday", "weekend"])
plt.show()

### TASK I — weekly accidents

From `acc` (one row per accident), build `weekly_acc`: the number of accidents per week, as a Series with
a weekly `DatetimeIndex`. Plot it as bars with a 4-week centred rolling mean on top, labelled axes and a title.

In [ ]:
# TASK I — your code here
weekly_acc = None

In [ ]:
# ✅ Self-check TASK I
assert weekly_acc is not None and isinstance(weekly_acc.index, pd.DatetimeIndex)
assert weekly_acc.index.freqstr.startswith("W"), "Use a weekly frequency."
assert 52 <= len(weekly_acc) <= 54
assert weekly_acc.sum() == len(acc), "Every accident counted once."
print("✅ TASK I passed")

## 5. Time zones

Timestamps without a zone are ambiguous. Use the IANA zone names from the standard library (`zoneinfo`;
pandas 3 no longer needs `pytz`).

In [ ]:
from zoneinfo import ZoneInfo

t = pd.Timestamp("2024-07-01 18:00").tz_localize(ZoneInfo("Europe/Madrid"))
print(t, "→", t.tz_convert("UTC"))

# Daylight saving: 2024-03-31 02:30 does not exist in Madrid
s = pd.Series(1, index=pd.DatetimeIndex(["2024-03-31 01:30", "2024-03-31 02:30", "2024-03-31 03:30"]))
print(s.tz_localize("Europe/Madrid", nonexistent="shift_forward").index)

## 6. Space and time

Doses per week by comarca as a heat map (rows sorted by total): space and time in one figure.

In [ ]:
com = pd.read_parquet(DATA / "examples/covid_vaccination_daily_comarca.parquet").dropna(subset=["comarca"])
com["date"] = pd.to_datetime(com["data"], format="ISO8601")
table = com.pivot_table(index="comarca", columns=pd.Grouper(key="date", freq="MS"), values="dosis", aggfunc="sum")
table = table.loc[table.sum(axis=1).sort_values(ascending=False).index]
share = table.div(table.sum(axis=1), axis=0)          # share of each comarca's doses per month

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(share, aspect="auto", cmap="viridis")
ax.set_yticks(range(len(share)), share.index.str.title(), fontsize=6)
ax.set_xticks(range(0, share.shape[1], 3), [d.strftime("%Y-%m") for d in share.columns[::3]],
              rotation=90, fontsize=7)
fig.colorbar(im, ax=ax, label="share of the comarca's doses")
ax.set_title("When did each comarca vaccinate?")
plt.show()

Normalising each row (share of its own total) lets you compare timing across comarques of very different sizes.

## 7. Fitting a curve (optional)

Fit a Gaussian to the third-dose campaign (weekly totals).

In [ ]:
from scipy.optimize import curve_fit

third = raw[raw["dosi"] == 3].groupby("date")["dosis"].sum().resample("W").sum()
third = third[third > 0]
t_days = (third.index - third.index[0]).days.to_numpy()


def gaussian(t, a, mu, sigma):
    return a * np.exp(-((t - mu) ** 2) / (2 * sigma**2))


params, _ = curve_fit(gaussian, t_days, third.to_numpy(), p0=[third.max(), t_days[third.argmax()], 30])
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(third.index, third, width=5, color="lightgrey", label="third doses per week")
ax.plot(third.index, gaussian(t_days, *params), color="darkred", label="Gaussian fit")
ax.set_ylabel("doses per week")
ax.legend()
plt.show()

Does a symmetric curve describe a vaccination campaign well? Look at the residuals before you trust a fit.

## 8. Your data

Build one time series of your project data at a sensible frequency, make gaps visible, and say in the
caption which aggregation or smoothing you used.

**Next:** `10_plot_redesign.ipynb` (Session 10).